In [9]:
# ==========================================
# PART 1: SMART INSTALLATION & IMPORTS
# ==========================================
import sys
import subprocess
import importlib.util
import os
import shutil

# 1. Install System Dependencies
print("Checking system dependencies...")
try:
    subprocess.run(["yum", "install", "-y", "unzip"], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "update"], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "unzip"], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
except Exception:
    pass

# 2. Python Library Installs
def install_if_missing(package_import_name, pip_package_name=None):
    if pip_package_name is None: pip_package_name = package_import_name
    if importlib.util.find_spec(package_import_name) is None:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_package_name])
        except subprocess.CalledProcessError:
            pass

# OCI Fix for OpenCV
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "opencv-python"], capture_output=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python-headless"])
    import cv2

install_if_missing("kaggle")
install_if_missing("pandas")
install_if_missing("torch")
install_if_missing("torchvision")
install_if_missing("sklearn", "scikit-learn") 

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# PART 2: DATASET LOADING (TOHWR ONLY)
# ==========================================

def setup_tohwr_dataset(target_dir="TOHWR_dataset"):
    print(f"\n--- Checking TOHWR Dataset ---")
    
    # 1. Persistence Check
    if os.path.exists(target_dir) and os.path.exists(os.path.join(target_dir, "train.csv")):
        print(f"TOHWR dataset already found at '{target_dir}'. Skipping download.")
        return target_dir

    # 2. Check Credentials
    kaggle_json = Path("kaggle.json")
    if not kaggle_json.exists() and not Path("~/.kaggle/kaggle.json").expanduser().exists():
        print("\n[ERROR] Missing 'kaggle.json'. Please upload it.")
        return None 

    # 3. Setup Config
    kaggle_dir = Path("~/.kaggle").expanduser()
    kaggle_dir.mkdir(exist_ok=True)
    if kaggle_json.exists():
        shutil.copy("kaggle.json", kaggle_dir / "kaggle.json")
    try: os.chmod(kaggle_dir / "kaggle.json", 0o600)
    except: pass

    # 4. Download
    print(f"Downloading TOHWR dataset...")
    try:
        subprocess.check_call(["kaggle", "datasets", "download", "-d", "sabarinathan/tamil-offline-handwritten-word-recognition", "-p", target_dir, "--unzip"])
    except:
        print("\n[ERROR] TOHWR download failed.")
        return None

    return target_dir

# ==========================================
# PART 3: CUSTOM DATA LOADER FOR TOHWR
# ==========================================

class TamilDataset(Dataset):
    """
    Custom Dataset to handle TOHWR's structure (CSV + Image Folder)
    """
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        
        # Encode string labels (Tamil words) to Integers
        self.label_encoder = LabelEncoder()
        # Assuming the CSV has a column 'label' or 'text'. Adjusting based on common Kaggle format.
        # If headers are different, we detect them.
        self.path_col = self.annotations.columns[0] # Usually filenames
        self.label_col = self.annotations.columns[1] # Usually labels
        
        # Handle non-string labels just in case
        self.annotations[self.label_col] = self.annotations[self.label_col].astype(str)
        self.annotations['encoded_label'] = self.label_encoder.fit_transform(self.annotations[self.label_col])
        
        self.classes = self.label_encoder.classes_

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        # Image path from CSV
        img_name = self.annotations.iloc[index, 0]
        img_path = os.path.join(self.root_dir, img_name)
        
        try:
            image = Image.open(img_path).convert("L") # Convert to Grayscale
        except (FileNotFoundError, OSError):
            # Fallback for missing/corrupt images to prevent crashing
            image = Image.new('L', (64, 64), color=0)
            
        label = torch.tensor(self.annotations.iloc[index, -1], dtype=torch.long) # Encoded label

        if self.transform:
            image = self.transform(image)

        return image, label

# ==========================================
# PART 4: MODEL (GAE-DCSSNN)
# ==========================================

class SoftSwish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

class GAE_Block(nn.Module):
    def __init__(self, channels, reduction=16):
        super(GAE_Block, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(1, channels // reduction), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // reduction), channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class GAE_DCSSNN(nn.Module):
    def __init__(self, num_classes):
        super(GAE_DCSSNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.ss1 = SoftSwish()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.ss2 = SoftSwish()
        self.gae = GAE_Block(64)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64*16*16, 128)
        self.ss3 = SoftSwish()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.ss1(self.conv1(x)))
        x = self.pool(self.ss2(self.conv2(x)))
        x = self.gae(x)
        x = self.flatten(x)
        x = self.ss3(self.fc1(x))
        return self.fc2(x)

# ==========================================
# PART 5: EXECUTION
# ==========================================

tohwr_path = setup_tohwr_dataset()

if not tohwr_path:
    print("\n[STOP] TOHWR Dataset failed to load.")
else:
    print("\n--- Preparing Data Loaders ---")
    
    # Locate train.csv
    train_csv_path = os.path.join(tohwr_path, "train.csv")
    
    # TOHWR structure usually has images inside a 'train' folder or mixed
    # We check common locations
    image_root = tohwr_path
    if os.path.exists(os.path.join(tohwr_path, "train")):
        image_root = os.path.join(tohwr_path, "train")
    
    if not os.path.exists(train_csv_path):
        print(f"[ERROR] Could not find 'train.csv' in {tohwr_path}")
    else:
        print(f"Found train.csv at: {train_csv_path}")
        print(f"Images expected in: {image_root}")

        transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
        ])

        try:
            # Load Custom Dataset
            full_dataset = TamilDataset(csv_file=train_csv_path, root_dir=image_root, transform=transform)
            
            # Split Train/Test
            train_size = int(0.8 * len(full_dataset))
            test_size = len(full_dataset) - train_size
            train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

            train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
            test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

            print(f"Training on {len(train_dataset)} images | Testing on {len(test_dataset)} images")
            print(f"Number of Classes (Words): {len(full_dataset.classes)}")
            
            # 3. TRAINING LOOP
            print("\n--- Starting Training (GAE-DCSSNN on TOHWR) ---")
            model = GAE_DCSSNN(num_classes=len(full_dataset.classes)).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001)

            EPOCHS = 5 # Reduced slightly for large dataset speed
            model.train()
            for epoch in range(EPOCHS):
                running_loss = 0.0
                correct = 0; total = 0
                for i, (images, labels) in enumerate(train_loader):
                    images, labels = images.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                    
                    running_loss += loss.item()
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
                    
                    # Print progress every 100 batches because dataset is huge
                    if i % 100 == 0:
                         print(f"   Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")
                
                print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {running_loss/len(train_loader):.4f} | Acc: {100 * correct / total:.2f}%")

            torch.save(model.state_dict(), "gae_dcssnn_tohwr.pth")
            print("Training Complete. Model Saved.")
            
        except Exception as e:
            print(f"[ERROR] Training setup failed: {e}")

Checking system dependencies...
Using device: cpu

--- Checking TOHWR Dataset ---
TOHWR dataset already found at 'TOHWR_dataset'. Skipping download.

--- Preparing Data Loaders ---
Found train.csv at: TOHWR_dataset/train.csv
Images expected in: TOHWR_dataset
Training on 60588 images | Testing on 15148 images
Number of Classes (Words): 11565

--- Starting Training (GAE-DCSSNN on TOHWR) ---
   Batch 0/1894 | Loss: 9.3563
   Batch 100/1894 | Loss: 9.3443
   Batch 200/1894 | Loss: 9.3874
   Batch 300/1894 | Loss: 9.3380
   Batch 400/1894 | Loss: 9.3653
   Batch 500/1894 | Loss: 9.3866
   Batch 600/1894 | Loss: 9.3766
   Batch 700/1894 | Loss: 9.4404
   Batch 800/1894 | Loss: 9.4318
   Batch 900/1894 | Loss: 9.4372
   Batch 1000/1894 | Loss: 9.4216
   Batch 1100/1894 | Loss: 9.4517
   Batch 1200/1894 | Loss: 9.4338
   Batch 1300/1894 | Loss: 9.5576
   Batch 1400/1894 | Loss: 9.4236
   Batch 1500/1894 | Loss: 9.5117
   Batch 1600/1894 | Loss: 9.4199
   Batch 1700/1894 | Loss: 9.4205
   Batch